# Comparativo del modulo #3 Sistema experto basado en tecnicas RAG para la base de conocimiento del sistema a nivel de usuario y las invocaciones al API de consultas de documentos..

# Integrantes:
- Diego Arámbulo
- Bryan Alvarado

In [1]:
!pip install -q pypdf langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import requests
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Definir las URLs
user_manual_url = "https://github.com/diegoarambulo/PROYECTO_TITULACION_GRUPO_1/blob/main/manuales%20de%20sistema/MANUAL_USUARIO_WEB.pdf"
tech_manual_url = "https://github.com/diegoarambulo/PROYECTO_TITULACION_GRUPO_1/blob/main/manuales%20de%20sistema/MANUAL_TECNICO_API.pdf"

# Convertir URLs de GitHub a formato raw
raw_user_url = user_manual_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
raw_tech_url = tech_manual_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

def download_and_extract(url, filename):
    response = requests.get(url)
    with open(filename, "wb") as f:
        f.write(response.content)
    reader = PdfReader(filename)
    text = ""
    for i, page in enumerate(reader.pages):
        text += page.extract_text() + f"\n\n[Metadata: file={filename}, page={i+1}]\n"
    return text

print("Descargando y extrayendo texto de los manuales...")
user_text = download_and_extract(raw_user_url, "manual_usuario.pdf")
tech_text = download_and_extract(raw_tech_url, "manual_tecnico.pdf")

# Dividir textos en chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

user_docs = [Document(page_content=chunk, metadata={"source": "manual_usuario"}) for chunk in text_splitter.split_text(user_text)]
tech_docs = [Document(page_content=chunk, metadata={"source": "manual_tecnico"}) for chunk in text_splitter.split_text(tech_text)]

print(f"Documentos generados: {len(user_docs)} (Usuario), {len(tech_docs)} (Técnico)")

Descargando y extrayendo texto de los manuales...
Documentos generados: 29 (Usuario), 5 (Técnico)


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from rank_bm25 import BM25Okapi

print("Generando embeddings e índices (esto puede tomar unos segundos)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Índices Vectoriales (FAISS)
user_vectorstore = FAISS.from_documents(user_docs, embeddings)
tech_vectorstore = FAISS.from_documents(tech_docs, embeddings)
combined_vectorstore = FAISS.from_documents(user_docs + tech_docs, embeddings)

# Índice Léxico (BM25) para Búsqueda Híbrida
combined_docs = user_docs + tech_docs
tokenized_corpus = [doc.page_content.split(" ") for doc in combined_docs]
bm25 = BM25Okapi(tokenized_corpus)

print("Índices creados exitosamente.")

Generando embeddings e índices (esto puede tomar unos segundos)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índices creados exitosamente.


In [4]:
import re
import time
import numpy as np

# 1. Self-Querying RAG (Simulado)
def self_querying_rag(query):
    # Simulación de extracción de metadatos por un LLM
    has_tech_keywords = any(kw in query.lower() for kw in ["api", "endpoint", "json", "código"])
    has_dates = re.search(r'\d{4}-\d{2}-\d{2}', query)

    # Filtrado pre-búsqueda
    vectorstore = tech_vectorstore if has_tech_keywords else user_vectorstore
    docs = vectorstore.similarity_search(query, k=2)
    return [d.page_content[:200] + "..." for d in docs]

# 2. Agentic RAG (Enrutador basado en reglas simulando agentes)
def agentic_rag(query):
    # El Agente Orquestador decide a qué agente llamar
    if "flujo" in query.lower() or "interfaz" in query.lower() or "usuario" in query.lower():
        agent = "Agente de Usuario"
        docs = user_vectorstore.similarity_search(query, k=2)
    else:
        agent = "Agente Técnico"
        docs = tech_vectorstore.similarity_search(query, k=2)
    return agent, [d.page_content[:200] + "..." for d in docs]

# 3. Hybrid Search (FAISS + BM25 con Reciprocal Rank Fusion)
def hybrid_search_rag(query):
    # Búsqueda Semántica
    semantic_docs = combined_vectorstore.similarity_search(query, k=5)

    # Búsqueda Léxica
    tokenized_query = query.split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    top_n_bm25 = np.argsort(bm25_scores)[::-1][:5]
    lexical_docs = [combined_docs[i] for i in top_n_bm25]

    # Fusión RRF simple
    combined_results = list({doc.page_content: doc for doc in semantic_docs + lexical_docs}.values())
    return [d.page_content[:200] + "..." for d in combined_results[:2]]

# 4. Semantic Routing
router_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
tech_route_vec = router_embeddings.embed_query("API REST endpoints json códigos de estado servidor base de datos")
user_route_vec = router_embeddings.embed_query("interfaz web botones flujo de usuario login registro clics")

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def semantic_routing_rag(query):
    query_vec = router_embeddings.embed_query(query)
    tech_sim = cosine_similarity(query_vec, tech_route_vec)
    user_sim = cosine_similarity(query_vec, user_route_vec)

    if tech_sim > user_sim:
        route = "Ruta Técnica"
        docs = tech_vectorstore.similarity_search(query, k=2)
    else:
        route = "Ruta de Usuario"
        docs = user_vectorstore.similarity_search(query, k=2)
    return route, [d.page_content[:200] + "..." for d in docs]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Consultas de prueba basadas en la solicitud
query_tech = "Busca en el manual técnico del API información sobre cual es el endpoint del servicio de busqueda de archivos."
query_user = "Explícame el flujo existente en el sistema web para crear un nuevo registro de usuario."

queries = [("Consulta Técnica", query_tech), ("Consulta de Usuario", query_user)]

for q_type, q_text in queries:
    print(f"\n{'='*50}\nEjecutando {q_type}: '{q_text}'\n{'='*50}")

    # 1. Self-Querying
    start = time.time()
    res_1 = self_querying_rag(q_text)
    print(f"[Self-Querying] ({(time.time()-start):.4f}s): Extrajo filtros y buscó.\nResultados: {len(res_1)}")
    print("===============================================")
    for i, res in enumerate(res_1):
        print(f"  - Res {i+1}: {res}")
    print("===============================================")
    # 2. Agentic RAG
    start = time.time()
    agent, res_2 = agentic_rag(q_text)
    print(f"\n[Agentic RAG] ({(time.time()-start):.4f}s): Enrutado por {agent}.\nResultados: {len(res_2)}")

    # 3. Hybrid Search
    start = time.time()
    res_3 = hybrid_search_rag(q_text)
    print(f"\n[Hybrid Search] ({(time.time()-start):.4f}s): Combinación Semántica + Léxica.\nResultados: {len(res_3)}")

    # 4. Semantic Routing
    start = time.time()
    route, res_4 = semantic_routing_rag(q_text)
    print(f"\n[Semantic Routing] ({(time.time()-start):.4f}s): Enrutado a {route}.\nResultados: {len(res_4)}")


Ejecutando Consulta Técnica: 'Busca en el manual técnico del API información sobre cual es el endpoint del servicio de busqueda de archivos.'
[Self-Querying] (0.0344s): Extrajo filtros y buscó.
Resultados: 2
  - Res 1: MANUAL_TECNICO_API.md 2026-04-04
1 / 3
Documentación de API: Búsqueda Global de
Documentos
Esta documentación detalla el uso del endpoint /documents/searchAll para la recuperación de entidades
de arch...
  - Res 2: [Metadata: file=manual_tecnico.pdf, page=2]
MANUAL_TECNICO_API.md 2026-04-04
3 / 3
  "documents": [] 
} 
4.3 Error del Sistema
Fallo interno durante el procesamiento de la solicitud.
Código HTTP: 500 ...

[Agentic RAG] (0.0306s): Enrutado por Agente Técnico.
Resultados: 2

[Hybrid Search] (0.0273s): Combinación Semántica + Léxica.
Resultados: 2

[Semantic Routing] (0.0562s): Enrutado a Ruta Técnica.
Resultados: 2

Ejecutando Consulta de Usuario: 'Explícame el flujo existente en el sistema web para crear un nuevo registro de usuario.'
[Self-Querying] (0.0269s)